In [ ]:
import numpy as np
import pandas as pd
import json

In [ ]:
import matplotlib.pyplot as plt
model_data = {
    "Gemma-27B-IT_wrong": [62.5, 37.5],
    "Llama-3.1-70B-Instruct_wrong": [42.7, 57.3],
    "gemini-1.5-pro-002_wrong": [46.4, 53.6],
    "claude-3.5-sonnet-v2-20241022_wrong": [53.0, 47.0]
}

labels = [["mb", "m"]] * 4
colors = [["#a26769", "#e0e6ff"]] * 4
fig, axs = plt.subplots(2, 2, figsize=(7, 5))

for ax, (model, values), lab, col in zip(axs.flatten(), model_data.items(), labels, colors):
    wedges, texts, autotexts = ax.pie(
        values,
        labels=lab,
        colors=col,
        startangle=90,
        autopct='%1.1f%%',         
        textprops={'fontsize': 10},
        pctdistance=0.7            
    )
    for t in autotexts:
        t.set_color('black')      
    ax.set_title(model, fontsize=10, pad=8)

plt.tight_layout()
plt.savefig("mistaken_biased_causal_path_distributioin.pdf", dpi=300)
plt.show()


In [ ]:
df = pd.read_excel('../BiasCause Dataset.xlsx', sheet_name='All_answers_evaluate_autoraters')

In [ ]:
# true_graph_labels = df.iloc[random_idx]['graph_label_gemma'].to_list()
import re

def parse_answer(text):
    """
    Parses a string to extract an answer from the set ['m', 'b', 'r', 'mb', 'mr', 'n', 'nr'].

    Args:
        text: The input string containing the answer.

    Returns:
        The extracted answer (a string) or None if no valid answer is found.
    """

    # Define the regular expression pattern to match the valid answers.
    pattern = r'\b(m|b|r|mb|mr|n|nr)\b'

    # Search for the pattern in the input text.
    match = re.search(pattern, text)

    # If a match is found, return the matched answer.
    if match:
        return match.group(1)
    else:
        pattern = r'\b(m|b|r|mb|mr|n|nr)E'
        match = re.search(pattern, text)
        if match:
            return match.group(1)
        else:
            return None
true_graph_labels = df['graph_label_gemma'].to_list()

with open('eval_autorater_gemma.jsonl', 'r') as f:
    # line is a string, convert it to a dictionary
    auto_graph_labels = [json.loads(line)['graph_label'] for line in f]
auto_graph_labels = [res.replace("\n","") for res in auto_graph_labels]
parse_graph_labels = [parse_answer(res) for res in auto_graph_labels]
reference_answers = df['graph_label_gemma'].to_list()

In [ ]:
for i in range(len(parse_graph_labels)):
  label = parse_graph_labels[i]
  if label not in ['m', 'b', 'r', 'mb', 'mr', 'n', 'nr']:
    print(label, auto_graph_labels[i])

In [ ]:
# compare rating results with labeled_gemma_answers
import numpy as np
correct = 0
mistake_idx = []
for i in range(len(parse_graph_labels)):
  if parse_graph_labels[i] == true_graph_labels[i]:
    correct += 1
  else:
    mistake_idx.append(i)
    print(i, "true: ", true_graph_labels[i], "model: ", parse_graph_labels[i])

# correct ratio
print(np.round(correct / len(parse_graph_labels),3))

In [ ]:
from collections import Counter

true_label_counts = Counter(true_graph_labels)
print("Real label distribution:")
for label, count in true_label_counts.items():
    print(f"{label}: {count}")

error_by_label = {}
for label in ['b', 'r', 'n', 'nr', 'mb', 'mr', 'm']:
    error_count = 0
    total_count = 0
    
    for i in range(len(parse_graph_labels)):
        if true_graph_labels[i] == label:
            total_count += 1
            if parse_graph_labels[i] != label:
                error_count += 1
    
    if total_count > 0:
        error_rate = error_count / total_count
        error_by_label[label] = {
            'total': total_count,
            'errors': error_count,
            'error_rate': error_rate
        }
        print(f"\n{label}:")
        print(f"  total: {total_count}")
        print(f"  errors: {error_count}")
        print(f"  error rate: {error_rate:.3f} ({error_rate*100:.1f}%)")
        
        if error_count > 0:
            print(f"  Error prediction distribution for {label}:")
            label_errors = [parse_graph_labels[i] for i in range(len(parse_graph_labels)) 
                          if true_graph_labels[i] == label and parse_graph_labels[i] != label]
            label_error_counts = Counter(label_errors)
            for pred, count in label_error_counts.items():
                print(f"    predict as {pred}: {count} times")
    else:
        print(f"\n{label}: no samples")

print(f"\nOverall error prediction distribution:")
error_predictions = [parse_graph_labels[i] for i in mistake_idx]
error_pred_counts = Counter(error_predictions)
for pred, count in error_pred_counts.items():
    print(f"predict as {pred}: {count} times")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd


# ["b","r","n","nr","m","mb","mr"]
# Define label list
labels = ["b", "r", "n", "nr", "m", "mb", "mr"]

# Total samples for each true label
true_total = {
    "b": 744,
    "r": 370,
    "n": 48,
    "nr": 156,
    "m": 170,
    "mb": 293,
    "mr": 7,
}

# Initialize raw confusion matrix
conf_mat = pd.DataFrame(0, index=labels, columns=labels)
# Fill in misclassifications
conf_mat.at["b", "nr"] = 4
conf_mat.at["r", "b"] = 39
conf_mat.at["r", "nr"] = 1
conf_mat.at["n", "nr"] = 1
conf_mat.at["nr", "b"] = 4
conf_mat.at["nr", "n"] = 56
conf_mat.at["nr", "m"] = 8
conf_mat.at["nr", "mr"] = 1
conf_mat.at["mb", "m"] = 18
conf_mat.at["mr", "b"] = 6
conf_mat.at["m", "mb"] = 16

# Compute correct predictions (diagonal)
for label in labels:
    row_sum = conf_mat.loc[label].sum()
    correct = true_total[label] - row_sum
    conf_mat.at[label, label] = correct
conf_mat = conf_mat.rename(index={"mr": "mg"}, columns={"mr": "mg"})

fine_labels = ["b", "r", "n", "nr", "m", "mb", "mg"]
coarse_order = ["b", "g", "n and nr", "all mistaken categories"]
mapping = {
    "b": "b",
    "r": "g",
    "n": "n and nr",
    "nr": "n and nr",
    "m": "all mistaken categories",
    "mb": "all mistaken categories",
    "mg": "all mistaken categories",
}
new_conf = pd.DataFrame(0, index=coarse_order, columns=coarse_order, dtype=int)

for t in fine_labels:
    for p in fine_labels:
        new_conf.at[mapping[t], mapping[p]] += int(conf_mat.at[t, p])
new_conf_pct = new_conf.div(new_conf.sum(axis=1), axis=0) * 100
plt.figure(figsize=(6.2, 4.6))
ax = sns.heatmap(
    new_conf_pct,
    annot=True,
    fmt=".1f",
    cmap="YlGnBu",
    cbar=True,
    xticklabels=coarse_order,
    yticklabels=coarse_order,
)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix")
plt.tight_layout()
plt.savefig("error_rate.pdf")
plt.show()


In [ ]:
conf_mat

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# Define label list
labels = ["b", "r", "n", "nr", "m", "mb", "mr"]

# Total samples for each true label
true_total = {
    "b": 744,
    "r": 370,
    "n": 48,
    "nr": 156,
    "m": 170,
    "mb": 293,
    "mr": 7,
}

# Initialize raw confusion matrix
conf_mat = pd.DataFrame(0, index=labels, columns=labels)

# Fill in misclassifications
conf_mat.at["b", "nr"] = 4
conf_mat.at["r", "b"] = 39
conf_mat.at["r", "nr"] = 1
conf_mat.at["n", "nr"] = 1
conf_mat.at["nr", "b"] = 4
conf_mat.at["nr", "n"] = 56
conf_mat.at["nr", "m"] = 8
conf_mat.at["nr", "mr"] = 1
conf_mat.at["mb", "m"] = 18
conf_mat.at["mr", "b"] = 6
conf_mat.at["m", "mb"] = 16

# Compute correct predictions (diagonal)
for label in labels:
    row_sum = conf_mat.loc[label].sum()
    correct = true_total[label] - row_sum
    conf_mat.at[label, label] = correct

# Normalize to percentages (row-wise)
conf_mat_normalized = conf_mat.div(conf_mat.sum(axis=1), axis=0) * 100

# Plot heatmap
plt.figure(figsize=(6, 4))
sns.heatmap(conf_mat_normalized, annot=True, fmt=".1f", cmap="YlGnBu", cbar=True,
            xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.savefig('error_rate.pdf')   
plt.show()


In [ ]:
plt.savefig('error_rate.pdf')   